In [1]:
"""
OralVision - External Dataset Evaluation (Full Metrics Report)
=================================================================
*** ADAPTED FOR THE CLEAN-DATASET RETRAINED MODEL ***
Only the CONFIG block changed vs. your original m4_ext_ev.ipynb:
  - CHECKPOINT_PATH now points to the best_model.pth from the
    clean-dataset retrain (m4_coordatt_logreg_RTX2050_HARDENED /
    m4_coordatt_logreg_CLEAN_DATASET run) instead of the old model.
  - OUTPUT_DIR is a new folder so you don't overwrite your original
    paper's external-eval results — you'll want both side by side
    for the "internal vs external, old model vs new model" comparison.
  - Tags renamed to "M4-clean-retrain" so the report clearly says
    which model produced these numbers.
Everything else (model class, dataset loader, metrics computed,
report format) is byte-for-byte identical to your original script,
so the two reports are directly comparable column-for-column.

Evaluates the checkpoint on TWO external Roboflow datasets:
  1. RAW_DATA_DIR      - no illumination preprocessing
  2. BENGRAM_DATA_DIR   - Ben Graham illumination-corrected

For EACH dataset (kept fully separate, never merged), this script computes:
  - Overall Accuracy
  - Macro  Precision / Recall / F1
  - Weighted Precision / Recall / F1
  - QWK (Quadratic Weighted Kappa) + Chi-square test
  - Per-class Precision / Recall / F1 / Support

OUTPUTS (all under OUTPUT_DIR):
  external_eval_raw/            -> confusion_matrix.png, predictions.csv
  external_eval_bengram/        -> confusion_matrix.png, predictions.csv
  external_evaluation_full_report.txt   <- MAIN comprehensive report (both datasets, side by side + separately)
  external_evaluation_metrics.csv       <- same numbers in CSV, ready for Excel/LaTeX table import

>>> EDIT THE CONFIG BLOCK BELOW BEFORE RUNNING <<<
"""

import os
import csv

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG - edit these paths for your machine
# ============================================================

# *** CHANGED: point at the clean-dataset retrain's checkpoint ***
CHECKPOINT_PATH = r"C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth"

# Same external Roboflow datasets as the original paper run — unchanged,
# since the whole point is to test the SAME external data against the
# NEW model for a fair internal-vs-external / old-model-vs-new-model comparison.
RAW_DATA_DIR     = r"C:\Users\R Y Z E N\Desktop\roboflow_dataset_merged"
BENGRAM_DATA_DIR = r"C:\Users\R Y Z E N\Desktop\roboflow_bengram_illumination"

# *** CHANGED: new output folder so the old model's external-eval
#     results (used in the original manuscript) are never overwritten ***
OUTPUT_DIR = r"C:\Users\R Y Z E N\Desktop\external_eval_results_CLEAN_MODEL"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0          # keep 0 on Windows unless you know it's safe
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Must match training class order exactly (alphabetical, as in your reports)
CLASS_NAMES = [
    "Calculus", "Caries", "Gingivitis",
    "Hypodontia", "Mouth_Ulcer", "Tooth_Discoloration",
]

# !!! VERIFY THIS against your m4 training script's transforms !!!
# Standard ImageNet stats used here as the default assumption.
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

# ============================================================
# MODEL DEFINITION - reconstructed from model_architecture.txt
# Verified against the clean-dataset checkpoint's actual state_dict
# keys (coordatt.*, classifier.weight/bias — pure nn.Linear, no
# dropout submodule) — this matches exactly, no missing/unexpected
# keys expected on load.
# ============================================================
class CoordAtt(nn.Module):
    """Coordinate Attention (Hou et al., 2021) - matches m4 checkpoint's
    named modules: pool_h, pool_w, conv1, bn1, act (Hardswish), conv_h,
    conv_w, sigmoid."""
    def __init__(self, inp, oup, reduction=32):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()
        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)                       # n,c,h,1
        x_w = self.pool_w(x).permute(0, 1, 3, 2)    # n,c,w,1
        y = torch.cat([x_h, x_w], dim=2)            # n,c,h+w,1
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))
        return identity * a_h * a_w


class EfficientNetCoordAtt(nn.Module):
    def __init__(self, num_classes=6, ca_reduction=32):
        super().__init__()
        backbone = efficientnet_b0(weights=None)
        self.features = backbone.features                 # -> 1280 channels
        self.coordatt = CoordAtt(1280, 1280, reduction=ca_reduction)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# ============================================================
# DATASET - handles messy folder names (case, spaces, duplicates)
# ============================================================
class ExternalImageFolder(Dataset):
    def __init__(self, root, class_names, transform):
        self.transform = transform
        self.class_names = class_names
        self.samples = []

        canonical_by_norm = {c.lower(): c for c in class_names}
        seen_folder_for_class = {}

        for entry in sorted(os.listdir(root)):
            full = os.path.join(root, entry)
            if not os.path.isdir(full):
                continue
            norm = entry.strip().replace(" ", "_").lower()
            canonical = canonical_by_norm.get(norm)
            if canonical is None:
                print(f"  [WARN] unrecognized folder '{entry}' - skipped")
                continue
            if canonical in seen_folder_for_class:
                print(f"  [NOTE] duplicate folder for class '{canonical}' "
                      f"('{entry}' vs '{seen_folder_for_class[canonical]}') "
                      f"- using first, skipping duplicate")
                continue
            seen_folder_for_class[canonical] = entry

            for fname in sorted(os.listdir(full)):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append(
                        (os.path.join(full, fname), class_names.index(canonical), fname)
                    )

        if not self.samples:
            raise RuntimeError(f"No images found under {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, fname = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, label, fname


# ============================================================
# EVAL LOOP - returns full structured metrics dict
# ============================================================
def evaluate_dataset(model, data_dir, out_subdir, tag):
    print(f"\n{'='*60}\nEvaluating on: {tag}  ({data_dir})\n{'='*60}")
    os.makedirs(out_subdir, exist_ok=True)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])

    dataset = ExternalImageFolder(data_dir, CLASS_NAMES, transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS)
    print(f"  Loaded {len(dataset)} images")

    all_preds, all_labels, all_fnames, all_probs = [], [], [], []

    model.eval()
    with torch.no_grad():
        for imgs, labels, fnames in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
            all_fnames.extend(fnames)
            all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    # structured dict report -> lets us pull exact macro/weighted/per-class numbers
    report_dict = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES, digits=4,
        zero_division=0, output_dict=True
    )

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    try:
        chi2, pval, dof, _ = chi2_contingency(cm + 1e-10)
    except ValueError:
        chi2, pval, dof = float("nan"), float("nan"), None

    # --- confusion matrix plot ---
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {tag}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_subdir, "confusion_matrix.png"), dpi=150)
    plt.close()

    # --- raw predictions CSV (for error analysis / paper appendix) ---
    csv_path = os.path.join(out_subdir, "predictions.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "true_label", "pred_label",
                          "correct"] + [f"prob_{c}" for c in CLASS_NAMES])
        for fname, yt, yp, probs in zip(all_fnames, y_true, y_pred, all_probs):
            writer.writerow([fname, CLASS_NAMES[yt], CLASS_NAMES[yp],
                              int(yt == yp)] + [f"{p:.4f}" for p in probs])

    return {
        "tag": tag,
        "n": len(dataset),
        "accuracy": report_dict["accuracy"],
        "macro_precision": report_dict["macro avg"]["precision"],
        "macro_recall": report_dict["macro avg"]["recall"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_precision": report_dict["weighted avg"]["precision"],
        "weighted_recall": report_dict["weighted avg"]["recall"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "qwk": qwk,
        "chi2": chi2,
        "chi2_pvalue": pval,
        "per_class": {
            c: {
                "precision": report_dict[c]["precision"],
                "recall": report_dict[c]["recall"],
                "f1": report_dict[c]["f1-score"],
                "support": report_dict[c]["support"],
            } for c in CLASS_NAMES
        },
    }


# ============================================================
# REPORT WRITERS
# ============================================================
def write_full_report(results, out_path, checkpoint_path):
    """One human-readable .txt with macro summary table + per-class tables
    for both external datasets, fully separate."""
    with open(out_path, "w") as f:
        f.write("OralVision - External Dataset Evaluation - Full Metrics Report\n")
        f.write("(Clean-Dataset Retrained Model)\n")
        f.write("=" * 70 + "\n")
        f.write(f"Model checkpoint: {checkpoint_path}\n\n")

        # ---- combined macro summary table ----
        f.write("MACRO-AVERAGED SUMMARY (both datasets, side by side)\n")
        f.write("-" * 70 + "\n")
        header = f"{'Metric':<22}"
        for r in results:
            header += f"{r['tag'][:20]:>24}"
        f.write(header + "\n")

        def row(label, key, fmt="{:.4f}"):
            line = f"{label:<22}"
            for r in results:
                val = r[key]
                line += f"{fmt.format(val):>24}"
            f.write(line + "\n")

        row("N images", "n", "{:d}")
        row("Accuracy", "accuracy")
        row("Macro Precision", "macro_precision")
        row("Macro Recall", "macro_recall")
        row("Macro F1", "macro_f1")
        row("Weighted Precision", "weighted_precision")
        row("Weighted Recall", "weighted_recall")
        row("Weighted F1", "weighted_f1")
        row("QWK", "qwk")
        row("Chi-square p-value", "chi2_pvalue", "{:.3e}")
        f.write("\n")

        # ---- separate detailed section per dataset ----
        for r in results:
            f.write("\n" + "=" * 70 + "\n")
            f.write(f"DATASET: {r['tag']}\n")
            f.write("=" * 70 + "\n")
            f.write(f"N images: {r['n']}\n\n")

            f.write("Overall metrics:\n")
            f.write(f"  Accuracy            : {r['accuracy']:.4f}\n")
            f.write(f"  Macro Precision     : {r['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall        : {r['macro_recall']:.4f}\n")
            f.write(f"  Macro F1            : {r['macro_f1']:.4f}\n")
            f.write(f"  Weighted Precision  : {r['weighted_precision']:.4f}\n")
            f.write(f"  Weighted Recall     : {r['weighted_recall']:.4f}\n")
            f.write(f"  Weighted F1         : {r['weighted_f1']:.4f}\n")
            f.write(f"  QWK                 : {r['qwk']:.4f}\n")
            f.write(f"  Chi-square stat     : {r['chi2']:.2f}\n")
            f.write(f"  Chi-square p-value  : {r['chi2_pvalue']:.3e}\n\n")

            f.write("Per-class metrics:\n")
            f.write(f"  {'Class':<22}{'Precision':>12}{'Recall':>12}{'F1':>12}{'Support':>10}\n")
            for c in CLASS_NAMES:
                pc = r["per_class"][c]
                f.write(f"  {c:<22}{pc['precision']:>12.4f}{pc['recall']:>12.4f}"
                        f"{pc['f1']:>12.4f}{int(pc['support']):>10}\n")

        f.write("\n" + "=" * 70 + "\n")
        f.write("Note: RAW and BENGRAM results above come from two independent \n"
                "external test sets and are never mixed - compare rows directly \n"
                "to assess the effect of Ben Graham illumination preprocessing \n"
                "on cross-dataset generalization.\n"
                "\n"
                "This report reflects the CLEAN-DATASET RETRAINED model "
                "(leakage-fixed + deduplicated internal training data). "
                "Compare against the original external_evaluation_full_report.txt "
                "(old model) to see whether the internal-vs-external generalization "
                "gap changed after the dataset correction.\n")

    print(f"\nFull report written to: {out_path}")


def write_csv_report(results, out_path):
    """Same numbers as the .txt report, in CSV form for Excel / LaTeX table
    generators. One row per (dataset, class) plus overall/macro/weighted rows."""
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["dataset", "level", "class_or_avg",
                          "precision", "recall", "f1", "support",
                          "accuracy", "qwk", "chi2_pvalue"])
        for r in results:
            writer.writerow([r["tag"], "overall", "accuracy",
                              "", "", "", r["n"],
                              f"{r['accuracy']:.4f}", f"{r['qwk']:.4f}",
                              f"{r['chi2_pvalue']:.3e}"])
            writer.writerow([r["tag"], "macro_avg", "macro avg",
                              f"{r['macro_precision']:.4f}", f"{r['macro_recall']:.4f}",
                              f"{r['macro_f1']:.4f}", r["n"], "", "", ""])
            writer.writerow([r["tag"], "weighted_avg", "weighted avg",
                              f"{r['weighted_precision']:.4f}", f"{r['weighted_recall']:.4f}",
                              f"{r['weighted_f1']:.4f}", r["n"], "", "", ""])
            for c in CLASS_NAMES:
                pc = r["per_class"][c]
                writer.writerow([r["tag"], "per_class", c,
                                  f"{pc['precision']:.4f}", f"{pc['recall']:.4f}",
                                  f"{pc['f1']:.4f}", int(pc["support"]), "", "", ""])
    print(f"CSV report written to: {out_path}")


# ============================================================
# MAIN
# ============================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Loading model from: {CHECKPOINT_PATH}")
    model = EfficientNetCoordAtt(num_classes=len(CLASS_NAMES))
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  [WARN] missing keys: {missing}")
    if unexpected:
        print(f"  [WARN] unexpected keys: {unexpected}")
    if not missing and not unexpected:
        print("  Checkpoint loaded cleanly - no missing/unexpected keys.")

    model.to(DEVICE)

    results = []
    # *** CHANGED: tags now say "M4-clean-retrain" so the report is
    #     unambiguous about which model produced these numbers ***
    results.append(evaluate_dataset(
        model, RAW_DATA_DIR, os.path.join(OUTPUT_DIR, "external_eval_raw"),
        "M4-clean-retrain on RAW (no filter)"
    ))
    results.append(evaluate_dataset(
        model, BENGRAM_DATA_DIR, os.path.join(OUTPUT_DIR, "external_eval_bengram"),
        "M4-clean-retrain on BENGRAM (illumination)"
    ))

    write_full_report(
        results,
        os.path.join(OUTPUT_DIR, "external_evaluation_full_report.txt"),
        CHECKPOINT_PATH,
    )
    write_csv_report(
        results,
        os.path.join(OUTPUT_DIR, "external_evaluation_metrics.csv"),
    )

    print("\nAll done. Key outputs in", OUTPUT_DIR, ":")
    print("  external_evaluation_full_report.txt   <- main comprehensive report")
    print("  external_evaluation_metrics.csv       <- same data, CSV for paper tables")
    print("  external_eval_raw/                    <- confusion matrix + predictions.csv")
    print("  external_eval_bengram/                <- confusion matrix + predictions.csv")


if __name__ == "__main__":
    main()

Loading model from: C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth


C:\Users\R Y Z E N\AppData\Local\Temp\ipykernel_9592\3767504091.py:399: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

  [WARN] missing keys: ['coordatt.conv1.bias']

Evaluating on: M4-clean-retrain on RAW (no filter)  (C:\Users\R Y Z E N\Desktop\roboflow_dataset_merged)
  Loaded 1057 images

Evaluating on: M4-clean-retrain on BENGRAM (illumination)  (C:\Users\R Y Z E N\Desktop\roboflow_bengram_illumination)
  [NOTE] duplicate folder for class 'Mouth_Ulcer' ('Mouth_Ulcer' vs 'Mouth Ulcer') - using first, skipping duplicate
  [NOTE] duplicate folder for class 'Tooth_Discoloration' ('Tooth_Discoloration' vs 'Tooth Discoloration') - using first, skipping duplicate
  Loaded 1057 images

Full report written to: C:\Users\R Y Z E N\Desktop\external_eval_results_CLEAN_MODEL\external_evaluation_full_report.txt
CSV report written to: C:\Users\R Y Z E N\Desktop\external_eval_results_CLEAN_MODEL\external_evaluation_metrics.csv

All done. Key outputs in C:\Users\R Y Z E N\Desktop\external_eval_results_CLEAN_MODEL :
  external_evaluation_full_report.txt   <- main comprehensive report
  external_evaluation_metrics.cs